# 2 — Core VTL rules

Hands-on notebook: run each cell with the **Trevas VTL** kernel.

**Prerequisites:** [1.1-trevas-intro.ipynb](1.1-trevas-intro.ipynb) · [1.2-vtl-intro.ipynb](1.2-vtl-intro.ipynb)

**Goals**
- Read and write basic VTL statements
- Use `:=` vs `<-`, comments, semicolons
- Understand unique names and component **roles**

**Next:** [3-first-operators.ipynb](3-first-operators.ipynb)


## General structure of a VTL statement <img style="float: right;" src="images/VTL_21.png" width="100" height="30">

Every top-level statement is a **transformation**:

```
name assignment_operator expression ;
```

| Part | Example |
|---|---|
| Result name (left) | `grades` |
| Assignment operator | `:=` or `<-` |
| Expression (right) | `loadCSV("data/students_grades.csv")` |
| Terminator | `;` |

Multiple statements in one cell are separated by semicolons. They are evaluated as a small **transformation scheme**.


## Temporary vs persistent assignment

| Operator | Name | Meaning |
|---|---|---|
| `:=` | non-persistent / temporary | Intermediate result inside the session |
| `<-` | persistent / *put* | Result intended for persistent storage |

Example:

```vtl
tmp := 1 + 2;           // temporary scalar
grades <- loadCSV("…"); // data set kept as persistent input
```

In Trevas Jupyter, both operators work in notebooks; choose `<-` when the data set represents a **declared input or output** of your pipeline, and `:=` for **working copies**.


In [ ]:
// Hello VTL in Trevas
message := "Hello VTL";
display := show(message);

## Comments <img style="float: right;" src="images/VTL_7.png" width="30" height="30">

VTL supports two comment styles (SDMX VTL 2.1 conventions):

```vtl
// single-line comment

/* multi-line
   comment */
```

Comments are ignored by the engine. Use them to document business intent.


In [ ]:
/* Training notebook — core rules */
greeting := "Ready to transform data"; // inline comment
greeting_out := show(greeting);

## One name, one artefact

In a VTL transformation scheme, each **name** identifies a single artefact. You cannot assign two different top-level transformations to the same name in the same scheme.

```vtl
// Valid — two different names
a := 1;
b := 2;

// Avoid in the same script — same name twice at top level
x := 1;
x := 2;   // second binding overwrites / conflicts depending on context
```

**Good practice:** use a **new name** for each logical step (`grades_raw`, `grades_clean`, `grades_agg`).

> **Trevas Jupyter note:** when you **re-run a cell**, the kernel clears the variables assigned in that cell before evaluating it again, so iterative notebook work remains practical. In a production VTL script, treat names as **immutable steps** in the transformation chain.

Assignments **inside** `[calc …]` blocks are different: they define components, not top-level artefacts.


## Component roles (critical for all operators)

Each column (component) has a **role** in the data structure:

| Role | VTL keyword in `calc` | Role in transformations |
|---|---|---|
| **Identifier** | `identifier` | Keys of the observation — always kept by `keep`/`drop` on measures |
| **Measure** | `measure` | Numeric or main values to aggregate, calculate, compare |
| **Attribute** | `attribute` | Descriptive fields linked to the observation |

Our training file `data/metadata_grades.json` documents:

- **Identifiers:** `Student_Id`, `Subject`, `Semester`
- **Measure:** `Grade`
- **Attributes:** `School`, `Teacher`

When you `loadCSV` in Trevas, columns start as raw fields. For realistic VTL behaviour, **declare roles** with a `calc` clause (shown below).


In [ ]:
// Load grades (comma-separated CSV)
grades_raw <- loadCSV("data/students_grades.csv?delimiter=,");

// Attach VTL roles — matches data/metadata_grades.json
grades := grades_raw
  [calc identifier Student_Id := Student_Id,
        identifier Subject := Subject,
        identifier Semester := Semester,
        measure Grade := Grade,
        attribute School := School,
        attribute Teacher := Teacher];

meta := showMetadata(grades);

Inspect the **metadata** output above: each component should show its role (`IDENTIFIER`, `MEASURE`, `ATTRIBUTE`).

Why roles matter for the operators in this module:
- **`keep` / `drop`** only accept measures and attributes — identifiers are always preserved
- **`rename`** changes component names while keeping values unchanged

The next notebook applies `keep`, `drop`, and `rename` on this structured data set.


In [ ]:
// Inspect row-level data
preview := show(grades);

## CSV formats — hands-on practice

Trevas add-on functions are documented in **[1.1-trevas-intro.ipynb](1.1-trevas-intro.ipynb)**. Below: load `ds1` / `ds2` and compare metadata **before** and **after** assigning roles with `calc`.


### Load `ds1` and `ds2` <img style="float: right;" src="images/VTL_27.png" width="30" height="80">

`ds1` uses the default semicolon delimiter; `ds2` is comma-separated with single quotes. Always assign `showMetadata`:

```vtl
meta := showMetadata(my_dataset);
```


In [ ]:
// ds1: semicolon-separated (Trevas default)
cities_raw <- loadCSV("data/ds1.csv");
cities_raw_meta := showMetadata(cities_raw);

In [ ]:
// ds2: comma-separated, single-quoted fields
nuts_raw <- loadCSV("data/ds2.csv?delimiter=%2c&quote=%27");
nuts_raw_meta := showMetadata(nuts_raw);

### Assign roles with `calc` — compare metadata before / after

Same pattern as for `students_grades`: declare identifiers and measures explicitly.


In [ ]:
cities := cities_raw
  [calc identifier city := city, measure nb1 := cast(nb1, integer)];

nuts := nuts_raw
  [calc identifier nuts3 := nuts3, measure nb2 := cast(nb2, number)];

cities_meta := showMetadata(cities);
nuts_meta   := showMetadata(nuts);

In [ ]:
cities_preview := show(cities);
row_count      := getSize(cities);

## Clause operators (preview)

Many VTL operators are written as **clauses** in square brackets after a data set:

```vtl
result := input [ operator arguments ];
```

Several clauses can be chained left to right:

```vtl
clean := grades [ drop School ] [ rename Grade to Score ];
```

We use this syntax extensively from **3-first-operators** onward.

---

Continue with **[3-first-operators.ipynb](3-first-operators.ipynb)**.
